In [21]:
#cell 1
# Set up the project environment and load the AURA dataset

from pathlib import Path
import random
import json

import cv2
import numpy as np
import pandas as pd

from google.colab import drive

# Connect Google Drive
drive.mount("/content/drive")

# Define project directories
PROJECT_DIR = Path("/content/drive/MyDrive/CAI421_Project")
DATA_DIR = PROJECT_DIR / "data" / "AURA" / "AURA"
ANNOTATIONS_DIR = DATA_DIR / "annotations"
REPORTS_DIR = PROJECT_DIR / "reports"
SPLITS_DIR = PROJECT_DIR / "splits"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Define the dataset scenes
SCENES = {
    "Scene_A": DATA_DIR / "scene_A",
    "Scene_B": DATA_DIR / "scene_B"
}

# Check the required dataset files
required_paths = [
    DATA_DIR / "README.md",
    ANNOTATIONS_DIR / "soft_labels.csv",
    ANNOTATIONS_DIR / "consensus_events.csv",
    SCENES["Scene_A"] / "normal_frames.csv",
    SCENES["Scene_A"] / "test_frames.csv",
    SCENES["Scene_A"] / "videos",
    SCENES["Scene_B"] / "normal_frames.csv",
    SCENES["Scene_B"] / "test_frames.csv",
    SCENES["Scene_B"] / "videos"
]

missing_paths = [
    str(path) for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing dataset paths:\n" + "\n".join(missing_paths)
    )

# Load annotation files
soft_labels = pd.read_csv(
    ANNOTATIONS_DIR / "soft_labels.csv"
)

consensus_events = pd.read_csv(
    ANNOTATIONS_DIR / "consensus_events.csv"
)

normal_frames = {}
test_frames = {}

for scene in ["A", "B"]:
    scene_dir = DATA_DIR / f"scene_{scene}"

    normal_frames[scene] = pd.read_csv(
        scene_dir / "normal_frames.csv"
    )

    test_frames[scene] = pd.read_csv(
        scene_dir / "test_frames.csv"
    )

# Validate the required columns
for scene in ["A", "B"]:
    if not {"video", "frame_idx"}.issubset(
        normal_frames[scene].columns
    ):
        raise ValueError(
            f"Invalid normal_frames.csv columns in Scene {scene}"
        )

    if not {"video", "frame_idx", "label"}.issubset(
        test_frames[scene].columns
    ):
        raise ValueError(
            f"Invalid test_frames.csv columns in Scene {scene}"
        )

if not {
    "scene", "video", "frame_idx", "soft_label"
}.issubset(soft_labels.columns):
    raise ValueError("Invalid soft_labels.csv columns.")

print("Dataset loaded successfully.")
print("Random seed:", SEED)
print("Soft-label records:", len(soft_labels))
print("Consensus events:", len(consensus_events))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully.
Random seed: 42
Soft-label records: 11709
Consensus events: 25


In [22]:
# cell 2
# Inspect videos and document dataset characteristics

video_records = []

expected_video_counts = {
    "Scene_A": 10,
    "Scene_B": 15
}

for scene_name, scene_dir in SCENES.items():

    video_dir = scene_dir / "videos"
    video_paths = sorted(video_dir.glob("*.mp4"))

    if len(video_paths) != expected_video_counts[scene_name]:
        raise ValueError(
            f"Unexpected video count in {scene_name}: "
            f"{len(video_paths)}"
        )

    for video_path in video_paths:

        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            raise ValueError(
                f"Could not open video: {video_path.name}"
            )

        frame_count = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        fps = float(
            cap.get(cv2.CAP_PROP_FPS)
        )

        width = int(
            cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        )

        height = int(
            cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        )

        cap.release()

        if frame_count <= 0 or not np.isfinite(fps) or fps <= 0:
            raise ValueError(
                f"Invalid video properties: {video_path.name}"
            )

        if width <= 0 or height <= 0:
            raise ValueError(
                f"Invalid resolution: {video_path.name}"
            )

        video_records.append({
            "scene": scene_name,
            "video": video_path.stem,
            "video_path": str(video_path),
            "frame_count": frame_count,
            "fps": fps,
            "width": width,
            "height": height,
            "resolution": f"{width}x{height}",
            "duration_sec": frame_count / fps,
            "size_mb": video_path.stat().st_size / (1024 ** 2)
        })

# Create the video inventory
inventory = pd.DataFrame(video_records)

if inventory.duplicated(
    subset=["scene", "video"]
).any():
    raise ValueError("Duplicate video identifiers were found.")

if len(inventory) != 25:
    raise ValueError(
        f"Expected 25 videos, found {len(inventory)}."
    )

# Save the detailed inventory
inventory.to_csv(
    REPORTS_DIR / "dataset_inventory.csv",
    index=False
)

# Summarize the dataset by scene
dataset_summary = inventory.groupby("scene").agg(
    video_count=("video", "nunique"),
    total_frames=("frame_count", "sum"),
    average_fps=("fps", "mean"),
    average_width=("width", "mean"),
    average_height=("height", "mean"),
    total_duration_sec=("duration_sec", "sum"),
    total_size_mb=("size_mb", "sum")
).reset_index()

dataset_summary.to_csv(
    REPORTS_DIR / "dataset_summary.csv",
    index=False
)

print("Dataset summary:")
display(dataset_summary.round(2))

print("\nVideo inventory:")
display(
    inventory[
        [
            "scene", "video", "frame_count", "fps",
            "resolution", "duration_sec", "size_mb"
        ]
    ].round(2)
)

print("\nTotal videos:", inventory["video"].nunique())
print("Total frames:", inventory["frame_count"].sum())

Dataset summary:


,scene,video_count,total_frames,average_fps,average_width,average_height,total_duration_sec,total_size_mb
0,Scene_A,10,12524,30.0,1920.0,1080.0,417.47,407.06
1,Scene_B,15,2559,15.0,1920.0,1080.0,170.60,31.04



Video inventory:


,scene,video,frame_count,fps,resolution,duration_sec,size_mb
0,Scene_A,v00,1309,30.0,1920x1080,43.63,34.31
1,Scene_A,v01,1303,30.0,1920x1080,43.43,45.94
2,Scene_A,v02,1507,30.0,1920x1080,50.23,36.24
3,Scene_A,v03,797,30.0,1920x1080,26.57,28.30
4,Scene_A,v04,1586,30.0,1920x1080,52.87,55.47
5,Scene_A,v05,1551,30.0,1920x1080,51.70,54.57
6,Scene_A,v06,531,30.0,1920x1080,17.70,16.75
7,Scene_A,v07,1382,30.0,1920x1080,46.07,53.89
8,Scene_A,v08,1061,30.0,1920x1080,35.37,36.33
9,Scene_A,v09,1497,30.0,1920x1080,49.90,45.26



Total videos: 25
Total frames: 15083


In [23]:
# Cell 3: Analyze label distribution and annotation quality

def normalize_scene(value):
    value = str(value).strip().lower()

    if value in {"a", "scene_a", "scene a"}:
        return "Scene_A"
    if value in {"b", "scene_b", "scene b"}:
        return "Scene_B"
    raise ValueError(f"Unknown scene name: {value}")


soft_labels["scene"] = soft_labels["scene"].apply(normalize_scene)

for scene in ["A", "B"]:
    normal_frames[scene]["scene"] = f"Scene_{scene}"
    test_frames[scene]["scene"] = f"Scene_{scene}"

# Keep the original test-label values visible instead of assuming labels are 0/1.
label_records = []
raw_test_label_records = []

for scene in ["A", "B"]:
    normal_df = normal_frames[scene]
    test_df = test_frames[scene]

    label_records.append({
        "scene": f"Scene_{scene}",
        "normal_training_frames": len(normal_df),
        "training_videos": normal_df["video"].astype(str).nunique(),
        "test_frames": len(test_df),
        "test_videos": test_df["video"].astype(str).nunique(),
        "test_label_missing": int(test_df["label"].isna().sum()),
        "unique_test_label_values": test_df["label"].dropna().astype(str).nunique()
    })

    value_counts = test_df["label"].astype("string").fillna("<MISSING>").value_counts()
    for label_value, count in value_counts.items():
        raw_test_label_records.append({
            "scene": f"Scene_{scene}",
            "test_label_value": str(label_value),
            "frame_count": int(count)
        })

label_distribution = pd.DataFrame(label_records)
raw_test_label_distribution = pd.DataFrame(raw_test_label_records)

print("Dataset frame counts and test-label availability:")
display(label_distribution)

print("\nActual values found in test_frames.csv label columns:")
display(raw_test_label_distribution)

label_distribution.to_csv(REPORTS_DIR / "label_distribution.csv", index=False)
raw_test_label_distribution.to_csv(
    REPORTS_DIR / "raw_test_label_distribution.csv", index=False
)

# Validate soft-label values and annotation records.
required_soft_label_columns = {"scene", "video", "frame_idx", "soft_label"}
if not required_soft_label_columns.issubset(soft_labels.columns):
    raise ValueError("soft_labels.csv is missing required columns.")

soft_labels["video"] = soft_labels["video"].astype(str)
soft_labels["frame_idx"] = pd.to_numeric(soft_labels["frame_idx"], errors="coerce")
soft_labels["soft_label"] = pd.to_numeric(soft_labels["soft_label"], errors="coerce")

soft_label_quality = pd.DataFrame([{
    "total_records": len(soft_labels),
    "missing_values": int(
        soft_labels[["scene", "video", "frame_idx", "soft_label"]].isna().sum().sum()
    ),
    "duplicate_frame_records": int(
        soft_labels.duplicated(subset=["scene", "video", "frame_idx"]).sum()
    ),
    "invalid_soft_labels": int(
        (
            (soft_labels["soft_label"] < 0)
            | (soft_labels["soft_label"] > 1)
        ).fillna(False).sum()
    )
}])

display(soft_label_quality)
soft_label_quality.to_csv(REPORTS_DIR / "soft_label_quality.csv", index=False)

# Summarize soft labels without converting them to hard classes.
soft_label_summary = soft_labels.groupby("scene").agg(
    labeled_frames=("frame_idx", "count"),
    unique_videos=("video", "nunique"),
    mean_soft_label=("soft_label", "mean"),
    median_soft_label=("soft_label", "median"),
    minimum_soft_label=("soft_label", "min"),
    maximum_soft_label=("soft_label", "max")
).reset_index()

soft_label_summary.to_csv(
    REPORTS_DIR / "soft_label_summary_by_scene.csv", index=False
)
print("\nSoft-label summary by scene:")
display(soft_label_summary.round(4))

# Calculate label coverage for each video.
label_coverage = soft_labels.groupby(["scene", "video"]).agg(
    labeled_frames=("frame_idx", "nunique"),
    mean_soft_label=("soft_label", "mean")
).reset_index()

label_coverage = inventory[["scene", "video", "frame_count"]].copy()
label_coverage["video"] = label_coverage["video"].astype(str)
label_coverage = label_coverage.merge(
    soft_labels.groupby(["scene", "video"]).agg(
        labeled_frames=("frame_idx", "nunique"),
        mean_soft_label=("soft_label", "mean")
    ).reset_index(),
    on=["scene", "video"],
    how="left"
)

label_coverage["labeled_frames"] = (
    label_coverage["labeled_frames"].fillna(0).astype(int)
)
label_coverage["coverage_percent"] = (
    label_coverage["labeled_frames"] / label_coverage["frame_count"] * 100
)

label_coverage.to_csv(REPORTS_DIR / "label_coverage.csv", index=False)

soft_labels["soft_label"].describe().to_csv(
    REPORTS_DIR / "soft_label_statistics.csv", header=["value"]
)

print("\nSoft-label coverage by video:")
display(label_coverage.round(2))

print("\nAnnotation quality:")
display(soft_label_quality)

if soft_label_quality.loc[0, "missing_values"] > 0:
    raise ValueError("Missing values found in required soft-label fields.")
if soft_label_quality.loc[0, "duplicate_frame_records"] > 0:
    raise ValueError("Duplicate (scene, video, frame_idx) soft-label records found.")
if soft_label_quality.loc[0, "invalid_soft_labels"] > 0:
    raise ValueError("Soft labels must be within [0, 1].")


Dataset frame counts and test-label availability:


,scene,normal_training_frames,training_videos,test_frames,test_videos,test_label_missing,unique_test_label_values
0,Scene_A,6516,7,4277,3,0,2
1,Scene_B,844,10,954,5,0,2



Actual values found in test_frames.csv label columns:


,scene,test_label_value,frame_count
0,Scene_A,normal,3394
1,Scene_A,anomalous,883
2,Scene_B,anomalous,628
3,Scene_B,normal,326


,total_records,missing_values,duplicate_frame_records,invalid_soft_labels
0,11709,0,0,0



Soft-label summary by scene:


,scene,labeled_frames,unique_videos,mean_soft_label,median_soft_label,minimum_soft_label,maximum_soft_label
0,Scene_A,9506,10,0.3009,0.0625,0.0,1.0
1,Scene_B,2203,15,0.5715,0.8125,0.0,1.0



Soft-label coverage by video:


,scene,video,frame_count,labeled_frames,mean_soft_label,coverage_percent
0,Scene_A,v00,1309,438,0.44,33.46
1,Scene_A,v01,1303,1303,0.27,100.00
2,Scene_A,v02,1507,1507,0.08,100.00
3,Scene_A,v03,797,797,0.37,100.00
4,Scene_A,v04,1586,1470,0.42,92.69
5,Scene_A,v05,1551,518,0.16,33.40
6,Scene_A,v06,531,509,0.39,95.86
7,Scene_A,v07,1382,406,0.36,29.38
8,Scene_A,v08,1061,1061,0.46,100.00
9,Scene_A,v09,1497,1497,0.24,100.00



Annotation quality:


,total_records,missing_values,duplicate_frame_records,invalid_soft_labels
0,11709,0,0,0


In [24]:
#cell 4 Prepare video frames, temporal sequences, augmentation, and data splits

IMAGE_SIZE = (224, 224)
SEQUENCE_LENGTH = 16

PREPROCESSING_CONFIG = {
    "image_width": IMAGE_SIZE[0],
    "image_height": IMAGE_SIZE[1],
    "color_format": "RGB",
    "pixel_range": [0.0, 1.0],
    "sequence_length": SEQUENCE_LENGTH,
    "random_seed": SEED,
    "augmentation": {
        "training_only": True,
        "horizontal_flip": False,
        "brightness_range": [0.9, 1.1],
        "contrast_range": [0.9, 1.1]
    }
}

# Preprocess a single frame
def preprocess_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    success, frame = cap.read()
    cap.release()

    if not success:
        raise ValueError(
            f"Could not read frame {frame_idx} from {video_path}"
        )

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    frame = cv2.resize(
        frame,
        IMAGE_SIZE,
        interpolation=cv2.INTER_AREA
    )

    return frame.astype(np.float32) / 255.0


# Apply consistent brightness and contrast changes to a sequence
def augment_sequence(sequence, rng):
    brightness = rng.uniform(0.9, 1.1)
    contrast = rng.uniform(0.9, 1.1)

    # Use one set of parameters for the entire sequence
    sequence_mean = sequence.mean(axis=(1, 2), keepdims=True)

    augmented = (
        (sequence - sequence_mean) * contrast
        + sequence_mean
    )

    augmented = augmented * brightness

    return np.clip(augmented, 0.0, 1.0).astype(np.float32)


# Read a video and create a fixed-length temporal sequence
def load_video_sequence(video_path, sequence_length=SEQUENCE_LENGTH,
                        augment=False, seed=SEED):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if frame_count <= 0:
        raise ValueError(f"Video has no readable frames: {video_path}")

    # Select frames evenly across the video
    frame_indices = np.linspace(
        0,
        frame_count - 1,
        num=min(sequence_length, frame_count),
        dtype=int
    )

    frames = [
        preprocess_frame(video_path, idx)
        for idx in frame_indices
    ]

    # Repeat the last frame if the video is shorter than the sequence
    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    sequence = np.stack(frames, axis=0)

    if augment:
        rng = np.random.default_rng(seed)
        sequence = augment_sequence(sequence, rng)

    return sequence


# Smoke-test preprocessing and augmentation on one real video before creating manifests.
sample_video_row = inventory.iloc[0]
sample_video_path = Path(sample_video_row["video_path"])

sample_sequence = load_video_sequence(
    sample_video_path,
    sequence_length=SEQUENCE_LENGTH,
    augment=False
)
sample_augmented_sequence = load_video_sequence(
    sample_video_path,
    sequence_length=SEQUENCE_LENGTH,
    augment=True,
    seed=SEED
)

expected_shape = (
    SEQUENCE_LENGTH,
    IMAGE_SIZE[1],
    IMAGE_SIZE[0],
    3
)

if sample_sequence.shape != expected_shape:
    raise ValueError(
        f"Unexpected sequence shape: {sample_sequence.shape}; expected {expected_shape}"
    )
if sample_augmented_sequence.shape != expected_shape:
    raise ValueError("Augmented sequence has an unexpected shape.")
if not np.isfinite(sample_sequence).all():
    raise ValueError("Preprocessed sequence contains NaN or infinite values.")
if sample_sequence.min() < 0 or sample_sequence.max() > 1:
    raise ValueError("Preprocessed pixels must be within [0, 1].")
if not np.isfinite(sample_augmented_sequence).all():
    raise ValueError("Augmented sequence contains NaN or infinite values.")
if sample_augmented_sequence.min() < 0 or sample_augmented_sequence.max() > 1:
    raise ValueError("Augmented pixels must be within [0, 1].")

print("Preprocessing smoke test passed.")
print("Sample video:", sample_video_row["video"])
print("Sequence shape:", sample_sequence.shape)
print("Pixel range:", float(sample_sequence.min()), "to", float(sample_sequence.max()))
print("Augmentation test passed; augmentation is only enabled when augment=True.")

# Save preprocessing settings
with open(
    REPORTS_DIR / "preprocessing_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(PREPROCESSING_CONFIG, file, indent=4)


# Collect the video IDs from the original annotations
normal_video_ids = {
    scene: sorted(
        normal_frames[scene]["video"].astype(str).unique()
    )
    for scene in ["A", "B"]
}

original_test_ids = {
    scene: sorted(
        test_frames[scene]["video"].astype(str).unique()
    )
    for scene in ["A", "B"]
}

all_video_ids = {
    scene: sorted(
        inventory.loc[
            inventory["scene"] == f"Scene_{scene}", "video"
        ].astype(str).unique()
    )
    for scene in ["A", "B"]
}

# Verify the original training and test videos do not overlap
for scene in ["A", "B"]:
    overlap = (
        set(normal_video_ids[scene])
        & set(original_test_ids[scene])
    )

    if overlap:
        raise ValueError(
            f"Original training/test overlap in Scene {scene}: {overlap}"
        )


# Split training videos into training and validation groups
def split_videos(video_ids, seed=SEED):
    video_ids = sorted(set(video_ids))

    if len(video_ids) < 2:
        raise ValueError(
            "At least two videos are required for train/validation."
        )

    shuffled = video_ids.copy()
    random.Random(seed).shuffle(shuffled)

    validation_count = max(
        1, round(len(shuffled) * 0.20)
    )
    validation_count = min(
        validation_count, len(shuffled) - 1
    )

    validation_ids = sorted(shuffled[:validation_count])
    training_ids = sorted(shuffled[validation_count:])

    return training_ids, validation_ids


# Define the four experiments in the research methodology
experiments = {
    "A_to_A": ("A", "A", "within_scene"),
    "B_to_B": ("B", "B", "within_scene"),
    "A_to_B": ("A", "B", "cross_scene"),
    "B_to_A": ("B", "A", "cross_scene")
}

video_split_records = []
frame_split_records = []

for experiment, (source, target, mode) in experiments.items():

    train_ids, val_ids = split_videos(
        normal_video_ids[source],
        seed=SEED
    )

    if mode == "within_scene":
        test_ids = original_test_ids[target]
    else:
        test_ids = all_video_ids[target]

    # Ensure the video groups are mutually exclusive
    assert not set(train_ids) & set(val_ids)
    assert not set(train_ids) & set(test_ids)
    assert not set(val_ids) & set(test_ids)

    # Record video-level assignments
    for split_name, video_ids, scene in [
        ("train", train_ids, source),
        ("validation", val_ids, source),
        ("test", test_ids, target)
    ]:
        for video in video_ids:
            video_split_records.append({
                "experiment": experiment,
                "split": split_name,
                "scene": f"Scene_{scene}",
                "video": video
            })

    # Select training frames from normal training annotations
    train_df = normal_frames[source].copy()
    train_df["scene"] = f"Scene_{source}"
    train_df = train_df[
        train_df["video"].astype(str).isin(train_ids)
    ].copy()
    train_df["split"] = "train"

    # Select validation frames from separate videos
    val_df = normal_frames[source].copy()
    val_df["scene"] = f"Scene_{source}"
    val_df = val_df[
        val_df["video"].astype(str).isin(val_ids)
    ].copy()
    val_df["split"] = "validation"

    # Select test frames according to the evaluation protocol
    if mode == "within_scene":
        test_df = test_frames[target].copy()
        test_df["scene"] = f"Scene_{target}"
    else:
        test_df = soft_labels[
            soft_labels["scene"] == f"Scene_{target}"
        ].copy()

    test_df = test_df[
        test_df["video"].astype(str).isin(test_ids)
    ].copy()
    test_df["split"] = "test"

    for df in [train_df, val_df, test_df]:
        df["experiment"] = experiment

    frame_split_records.extend([
        train_df,
        val_df,
        test_df
    ])


# Combine split records
video_splits = pd.DataFrame(video_split_records)

frame_splits = pd.concat(
    frame_split_records,
    ignore_index=True,
    sort=False
)

# Save the complete split manifests
video_splits.to_csv(
    SPLITS_DIR / "video_splits.csv",
    index=False
)

frame_splits.to_csv(
    SPLITS_DIR / "frame_splits.csv",
    index=False
)

# Save separate files for each experiment and split
for experiment in experiments:
    for split_name in ["train", "validation", "test"]:

        subset = frame_splits[
            (frame_splits["experiment"] == experiment)
            & (frame_splits["split"] == split_name)
        ]

        subset.to_csv(
            SPLITS_DIR / f"{experiment}_{split_name}.csv",
            index=False
        )

print("Video-level split summary:")

display(
    video_splits.groupby(
        ["experiment", "split", "scene"]
    ).agg(
        video_count=("video", "nunique")
    ).reset_index()
)

print("\nFrame-level split summary:")

display(
    frame_splits.groupby(
        ["experiment", "split"]
    ).agg(
        frame_count=("frame_idx", "count"),
        video_count=("video", "nunique")
    ).reset_index()
)

# Report which test records have soft labels available.
test_rows = frame_splits[frame_splits["split"] == "test"].copy()
if "soft_label" in test_rows.columns:
    if "label" not in test_rows.columns:
        test_rows["label"] = np.nan
    test_rows["has_evaluation_label"] = (
        test_rows["soft_label"].notna() | test_rows["label"].notna()
    )
    test_label_coverage = test_rows.groupby("experiment").agg(
        test_records=("frame_idx", "count"),
        records_with_evaluation_label=("has_evaluation_label", "sum"),
        soft_label_records=("soft_label", "count"),
        hard_label_records=("label", "count"),
        test_videos=("video", "nunique")
    ).reset_index()
    test_label_coverage["label_coverage_percent"] = (
        test_label_coverage["records_with_evaluation_label"]
        / test_label_coverage["test_records"] * 100
    )
else:
    test_label_coverage = pd.DataFrame()

if not test_label_coverage.empty:
    test_label_coverage.to_csv(
        REPORTS_DIR / "test_split_label_coverage.csv", index=False
    )
    print("\nTest split label coverage:")
    display(test_label_coverage.round(2))

print("\nPreprocessing and splitting completed.")

Preprocessing smoke test passed.
Sample video: v00
Sequence shape: (16, 224, 224, 3)
Pixel range: 0.0 to 0.9529411792755127
Augmentation test passed; augmentation is only enabled when augment=True.
Video-level split summary:


,experiment,split,scene,video_count
0,A_to_A,test,Scene_A,3
1,A_to_A,train,Scene_A,6
2,A_to_A,validation,Scene_A,1
3,A_to_B,test,Scene_B,15
4,A_to_B,train,Scene_A,6
5,A_to_B,validation,Scene_A,1
6,B_to_A,test,Scene_A,10
7,B_to_A,train,Scene_B,8
8,B_to_A,validation,Scene_B,2
9,B_to_B,test,Scene_B,5



Frame-level split summary:


,experiment,split,frame_count,video_count
0,A_to_A,test,4277,3
1,A_to_A,train,5093,6
2,A_to_A,validation,1423,1
3,A_to_B,test,2203,15
4,A_to_B,train,5093,6
5,A_to_B,validation,1423,1
6,B_to_A,test,9506,10
7,B_to_A,train,713,8
8,B_to_A,validation,131,2
9,B_to_B,test,954,5



Test split label coverage:


,experiment,test_records,records_with_evaluation_label,soft_label_records,hard_label_records,test_videos,label_coverage_percent
0,A_to_A,4277,4277,0,4277,3,100.0
1,A_to_B,2203,2203,2203,0,15,100.0
2,B_to_A,9506,9506,9506,0,10,100.0
3,B_to_B,954,954,0,954,5,100.0



Preprocessing and splitting completed.


In [25]:
#cell 5 Validate data splits and prepare the team handoff

leakage_records = []

for experiment in experiments:

    current = video_splits[
        video_splits["experiment"] == experiment
    ]

    train_ids = set(
        current.loc[current["split"] == "train", "video"]
    )

    val_ids = set(
        current.loc[current["split"] == "validation", "video"]
    )

    test_ids = set(
        current.loc[current["split"] == "test", "video"]
    )

    train_val_overlap = train_ids & val_ids
    train_test_overlap = train_ids & test_ids
    val_test_overlap = val_ids & test_ids

    current_frames = frame_splits[
        frame_splits["experiment"] == experiment
    ]

    frame_assignment_errors = 0

    for split_name, allowed_videos in [
        ("train", train_ids),
        ("validation", val_ids),
        ("test", test_ids)
    ]:
        subset = current_frames[
            current_frames["split"] == split_name
        ]

        invalid_rows = subset[
            ~subset["video"].astype(str).isin(allowed_videos)
        ]

        frame_assignment_errors += len(invalid_rows)

    leakage_records.append({
        "experiment": experiment,
        "train_videos": len(train_ids),
        "validation_videos": len(val_ids),
        "test_videos": len(test_ids),
        "train_validation_overlap": len(train_val_overlap),
        "train_test_overlap": len(train_test_overlap),
        "validation_test_overlap": len(val_test_overlap),
        "frame_assignment_errors": frame_assignment_errors,
        "passed": (
            len(train_val_overlap) == 0
            and len(train_test_overlap) == 0
            and len(val_test_overlap) == 0
            and frame_assignment_errors == 0
        )
    })

leakage_report = pd.DataFrame(leakage_records)

leakage_report.to_csv(
    REPORTS_DIR / "leakage_check.csv",
    index=False
)

print("Leakage validation:")
display(leakage_report)

if not leakage_report["passed"].all():
    raise ValueError(
        "Split validation failed. Review leakage_check.csv."
    )


# Check that no identical frame record is assigned more than once per experiment.
duplicate_frame_rows = frame_splits.duplicated(
    subset=["experiment", "scene", "video", "frame_idx", "split"]
).sum()
if duplicate_frame_rows:
    raise ValueError(
        f"Found {duplicate_frame_rows} duplicate frame assignments within the same split."
    )

# Ensure every frame row belongs to a video assigned to the same scene and split.
scene_assignment_errors = []
for row in frame_splits[
    ["experiment", "split", "scene", "video"]
].drop_duplicates().itertuples(index=False):
    experiment, split_name, scene, video = row
    assigned = video_splits[
        (video_splits["experiment"] == experiment)
        & (video_splits["split"] == split_name)
        & (video_splits["scene"] == scene)
        & (video_splits["video"].astype(str) == str(video))
    ]
    if assigned.empty:
        scene_assignment_errors.append({
            "experiment": experiment,
            "split": split_name,
            "scene": scene,
            "video": video
        })

scene_assignment_report = pd.DataFrame(scene_assignment_errors)
scene_assignment_report.to_csv(
    REPORTS_DIR / "scene_assignment_errors.csv", index=False
)
if not scene_assignment_report.empty:
    display(scene_assignment_report)
    raise ValueError("Frame rows do not match the video-level split manifest.")

# Validate frame references against the video inventory
video_lookup = inventory.set_index(
    ["scene", "video"]
)["frame_count"].to_dict()

reference_errors = []

for row in frame_splits[
    ["scene", "video", "frame_idx", "experiment", "split"]
].itertuples(index=False):

    scene, video, frame_idx, experiment, split_name = row

    frame_count = video_lookup.get(
        (scene, str(video))
    )

    if frame_count is None:
        reference_errors.append({
            "experiment": experiment,
            "split": split_name,
            "scene": scene,
            "video": video,
            "frame_idx": frame_idx,
            "issue": "Video not found"
        })

    elif pd.isna(frame_idx) or not (
        0 <= frame_idx < frame_count
    ):
        reference_errors.append({
            "experiment": experiment,
            "split": split_name,
            "scene": scene,
            "video": video,
            "frame_idx": frame_idx,
            "issue": "Invalid frame index"
        })

reference_report = pd.DataFrame(
    reference_errors,
    columns=[
        "experiment", "split", "scene",
        "video", "frame_idx", "issue"
    ]
)

reference_report.to_csv(
    REPORTS_DIR / "frame_reference_errors.csv",
    index=False
)

if not reference_report.empty:
    display(reference_report.head(20))
    raise ValueError(
        "Invalid frame references were detected."
    )


# Save the complete pipeline configuration
pipeline_config = {
    "dataset": "AURA",
    "random_seed": SEED,
    "split_unit": "video",
    "validation_ratio": 0.20,
    "validation_source": "normal_frames.csv videos held out from training",
    "within_scene_test_source": "test_frames.csv",
    "cross_scene_test_source": "soft_labels.csv labeled frames only",
    "soft_label_interpretation": "continuous annotation score in [0, 1]; do not assume binary labels",
    "image_size": list(IMAGE_SIZE),
    "sequence_length": SEQUENCE_LENGTH,
    "color_format": "RGB",
    "pixel_range": [0.0, 1.0],
    "augmentation": PREPROCESSING_CONFIG["augmentation"],
    "experiments": list(experiments.keys()),
    "video_splits_file": str(
        SPLITS_DIR / "video_splits.csv"
    ),
    "frame_splits_file": str(
        SPLITS_DIR / "frame_splits.csv"
    )
}

with open(
    REPORTS_DIR / "dataset_pipeline_config.json",
    "w",encoding="utf-8"
) as file:
    json.dump(pipeline_config, file, indent=4)


# Create a handoff guide for Members 2 and 4
handoff_text = """
CAI 421 - Student 1 Dataset Preparation

Dataset:
AURA underwater video anomaly detection dataset.

Main reports:
- dataset_inventory.csv
- dataset_summary.csv
- label_distribution.csv
- raw_test_label_distribution.csv
- soft_label_summary_by_scene.csv
- soft_label_quality.csv
- label_coverage.csv
- leakage_check.csv
- frame_reference_errors.csv
- scene_assignment_errors.csv
- test_split_label_coverage.csv
- preprocessing_config.json
- dataset_pipeline_config.json

Split files:
- splits/video_splits.csv
- splits/frame_splits.csv
- Separate train, validation, and test CSV files for each experiment.

Preprocessing:
Frames are converted from BGR to RGB, resized, and normalized
to [0, 1]. Temporal sequences are created using a fixed sequence length.
Brightness and contrast augmentation is intended for training only.

Experiments:
A_to_A: Train and validate on Scene A normal training videos;
test on the original Scene A test videos.

B_to_B: Train and validate on Scene B normal training videos;
test on the original Scene B test videos.

A_to_B: Train and validate using Scene A normal training videos;
test on labeled frames available in Scene B.

B_to_A: Train and validate using Scene B normal training videos;
test on labeled frames available in Scene A.

Reproducibility:
Random seed: 42.
Splitting is performed at video level.

Important:
The split manifests contain annotated frame records. They do not
contain precomputed image tensors or complete video sequences.
Use the preprocessing functions and split manifests consistently.

Cross-scene evaluation uses the labels available in soft_labels.csv.
Review label_coverage.csv and test_split_label_coverage.csv before interpreting evaluation results.
The soft labels are continuous scores in [0, 1]; choose and document any
threshold only if the evaluation metric requires binary labels. The original
test_frames.csv label values are reported as observed in raw_test_label_distribution.csv.
"""

(REPORTS_DIR / "README_student1.txt").write_text(
    handoff_text.strip(),
    encoding="utf-8"
)

print("SUCCESS: Video-level split checks passed.")
print("SUCCESS: Frame-reference checks passed.")
print("Pipeline configuration saved.")
print("Team handoff guide saved.")
print("Reports directory:", REPORTS_DIR)
print("Splits directory:", SPLITS_DIR)

Leakage validation:


,experiment,train_videos,validation_videos,test_videos,train_validation_overlap,train_test_overlap,validation_test_overlap,frame_assignment_errors,passed
0,A_to_A,6,1,3,0,0,0,0,True
1,B_to_B,8,2,5,0,0,0,0,True
2,A_to_B,6,1,15,0,0,0,0,True
3,B_to_A,8,2,10,0,0,0,0,True


SUCCESS: Video-level split checks passed.
SUCCESS: Frame-reference checks passed.
Pipeline configuration saved.
Team handoff guide saved.
Reports directory: /content/drive/MyDrive/CAI421_Project/reports
Splits directory: /content/drive/MyDrive/CAI421_Project/splits
